# 02 · Tests

Checks for the code in `01_entity_model.ipynb`. Run All; every check prints
`passed` / `[ok]`, and a failing check stops with an error.

1. **Unit tests**: helpers and dataset loaders on tiny examples (seconds)
2. **Real data check**: loads the actual dataset files and counts documents
3. **Smoke test**: builds the model, checks shapes against the architecture
   diagram and overfits one sentence (about a minute; downloads BERT the first time)

In [ ]:
# Load the model code from 01_entity_model.ipynb (its cells tagged "definitions"):
# settings, helpers, dataset loaders, the model and the training functions.
# Nothing is trained here. Keep this notebook in the same folder as notebook 01.
import json as _json
from pathlib import Path as _Path

_nb = _json.loads(_Path("01_entity_model.ipynb").read_text(encoding="utf-8"))
for _cell in _nb["cells"]:
    if _cell["cell_type"] == "code" and "definitions" in _cell["metadata"].get("tags", []):
        exec("".join(_cell["source"]), globals())
print("Loaded the model code from 01_entity_model.ipynb")

## 1a. Unit tests: helpers

Tokenising, the 52 candidate spans of the diagram's example, character offsets,
training targets, choosing mentions (single- and multi-label), strict F1.

In [ ]:
"""Unit tests for ee_utils (pure Python, no PyTorch). Run: python3 test_ee_utils.py"""


TEXT = "Maya founded Northstar Labs. The company makes biosensors."
LABELS = ["person", "organization", "method"]


def test_tokenize_matches_diagram():
    words, offsets, sents = tokenize_with_offsets(TEXT)
    assert words == ["Maya", "founded", "Northstar", "Labs", ".",
                     "The", "company", "makes", "biosensors", "."]
    # 'Northstar Labs' = chars 13-27, 'The company' = chars 29-40 (as in the diagram)
    assert (offsets[2][0], offsets[3][1]) == (13, 27)
    assert (offsets[5][0], offsets[6][1]) == (29, 40)
    assert sents == [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]


def test_tokenize_clitics_abbreviations_and_sentences():
    words, _, sents = tokenize_with_offsets(
        "Germany's leader met Dr. Merkel in the U.S. today. It cost 1,000 euros (3.5%). Fine!")
    assert words[:2] == ["Germany", "'s"]
    assert "U.S." in words and "1,000" in words and "3.5" in words
    assert sents[words.index("Merkel")] == 0            # 'Dr.' does not end a sentence
    assert sents[words.index("It")] == 1                # 'today.' does
    assert sents[words.index("Fine")] == 2              # ')' then '.' does
    cut_words, _, _ = tokenize_with_offsets("ZDF-Politbarometer poll", split_points=[0, 3])
    assert cut_words[0] == "ZDF"


def test_span_count_matches_diagram():
    assert len(enumerate_spans(10, max_width=8)) == 52            # diagram's count
    _, _, sents = tokenize_with_offsets(TEXT)
    within = enumerate_spans(10, max_width=8, sentence_ids=sents)
    assert all(sents[s] == sents[e] for s, e in within)
    assert len(within) == 30                                     # 15 per 5-token sentence


def test_char_to_word_span():
    _, offsets, _ = tokenize_with_offsets(TEXT)
    assert char_to_word_span(13, 27, offsets) == (2, 3)
    assert char_to_word_span(29, 40, offsets) == (5, 6)
    assert char_to_word_span(1000, 1001, offsets) is None


def test_targets_and_stats():
    spans = enumerate_spans(10, 8)
    gold = [
        {"start": 0, "end": 0, "label": "person"},
        {"start": 2, "end": 3, "label": "organization"},
        {"start": 5, "end": 6, "label": "organization"},
        {"start": 0, "end": 9, "label": "organization"},   # wider than 8 tokens
        {"start": 8, "end": 8, "label": "material"},        # not in this label set
    ]
    targets, stats = build_entity_targets(spans, gold, LABELS)
    assert len(targets) == 52 and len(targets[0]) == 3
    assert targets[spans.index((2, 3))] == [0.0, 1.0, 0.0]
    assert sum(map(sum, targets)) == 3
    assert stats == {"gold": 5, "matched": 3, "too_long_or_filtered": 1, "unknown_label": 1}


def test_decode_keeps_nested_and_char_offsets():
    doc = document_from_text(TEXT, LABELS)
    spans = enumerate_spans(len(doc.words), 8)
    probs = [[0.01, 0.02, 0.02] for _ in spans]
    probs[spans.index((0, 0))] = [0.95, 0.03, 0.01]   # Maya -> person
    probs[spans.index((2, 3))] = [0.04, 0.91, 0.02]   # Northstar Labs -> organization
    probs[spans.index((3, 3))] = [0.02, 0.60, 0.01]   # nested 'Labs' also kept
    probs[spans.index((0, 1))] = [0.40, 0.30, 0.10]   # 'Maya founded' below 0.5
    mentions = decode_mentions(spans, probs, LABELS, doc)
    assert [(m["text"], m["label"]) for m in mentions] == [
        ("Maya", "person"), ("Northstar Labs", "organization"), ("Labs", "organization")]
    m2 = mentions[1]
    assert (m2["start"], m2["end"], m2["char_start"], m2["char_end"]) == (2, 3, 13, 27)
    assert m2["span_idx"] == spans.index((2, 3))


def test_decode_multi_label():
    doc = document_from_text(TEXT, LABELS)
    doc.multi_label = True
    spans = enumerate_spans(len(doc.words), 8)
    probs = [[0.01, 0.02, 0.02] for _ in spans]
    probs[spans.index((2, 3))] = [0.04, 0.91, 0.70]   # two labels above 0.5
    mentions = decode_mentions(spans, probs, LABELS, doc)
    assert [(m["text"], m["label"]) for m in mentions] == [
        ("Northstar Labs", "organization"), ("Northstar Labs", "method")]
    assert mentions[0]["span_idx"] == mentions[1]["span_idx"]


def test_entity_prf_is_strict():
    gold = [[{"start": 0, "end": 0, "label": "person"},
             {"start": 2, "end": 3, "label": "organization"}]]
    pred = [[{"start": 0, "end": 0, "label": "person"},          # correct
             {"start": 2, "end": 2, "label": "organization"},    # wrong boundary
             {"start": 5, "end": 6, "label": "person"}]]         # wrong span
    s = entity_prf(pred, gold)
    assert round(s["precision"], 2) == 33.33 and s["recall"] == 50.0
    assert round(s["f1"], 2) == 40.0

for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn) and _fn.__module__ == __name__:
        _fn()
        print("passed:", _name)
        del globals()[_name]          # so the next test cell only runs its own tests

## 1b. Unit tests: dataset loaders

Each loader on a tiny file in that dataset's format.

In [ ]:
"""Tests the loaders on tiny files in each dataset's format (no PyTorch).
Run: python3 test_datasets.py"""

import json
import os
import tempfile



def test_scierc():
    ex = {"doc_key": "X1", "sentences": [["We", "use", "neural", "nets", "."],
                                         ["They", "help", "parsing", "."]],
          "ner": [[[2, 3, "Method"]], [[7, 7, "Task"]]]}
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
        f.write(json.dumps(ex) + "\n")
    doc = load_scierc(f.name)[0]
    assert doc.words[7] == "parsing" and doc.sentence_ids[7] == 1
    assert doc.gold_mentions == [{"start": 2, "end": 3, "label": "method"},
                                 {"start": 7, "end": 7, "label": "task"}]
    assert "other scientific term" in doc.entity_labels


def test_redocred():
    ex = [{"title": "T", "sents": [["Maya", "founded", "Northstar", "Labs", "."],
                                    ["The", "company", "grew", "."]],
           "vertexSet": [[{"name": "Maya", "sent_id": 0, "pos": [0, 1], "type": "PER"}],
                         [{"name": "Northstar Labs", "sent_id": 0, "pos": [2, 4], "type": "ORG"},
                          {"name": "The company", "sent_id": 1, "pos": [0, 2], "type": "ORG"}]]}]
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
        json.dump(ex, f)
    doc = load_redocred(f.name)[0]
    got = [(m["start"], m["end"], m["label"]) for m in doc.gold_mentions]
    assert got == [(0, 0, "person"), (2, 3, "organization"), (5, 6, "organization")]
    assert doc.entity_labels == sorted(["person", "organization", "location", "time",
                                        "number", "miscellaneous"])


def test_dwie():
    text = "Angela Merkel visited Berlin."
    ex = {"id": "DW_1", "tags": ["all", "train"], "content": text,
          "mentions": [{"begin": 0, "end": 13, "text": "Angela Merkel", "concept": 0},
                       {"begin": 22, "end": 28, "text": "Berlin", "concept": 1}],
          "concepts": [{"concept": 0, "tags": ["type::entity", "type::person"]},
                       {"concept": 1, "tags": ["type::location", "type::gpe2"]}]}
    d = tempfile.mkdtemp()
    with open(os.path.join(d, "DW_1.json"), "w") as f:
        json.dump(ex, f)
    doc = load_dwie(d, split="train")[0]
    got = [(m["start"], m["end"], m["label"]) for m in doc.gold_mentions]
    # multi-label: every type tag except the generic 'entity'
    assert got == [(0, 1, "person"), (3, 3, "location"), (3, 3, "gpe2")]
    assert doc.multi_label and doc.entity_labels == ["gpe2", "location", "person"]
    assert load_dwie(d, split="test") == []

for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn) and _fn.__module__ == __name__:
        _fn()
        print("passed:", _name)
        del globals()[_name]          # so the next test cell only runs its own tests

## 2. Real data check

Loads the real files for the datasets in the settings. For Re-DocRED expect
3,053 / 500 / 500 documents.

In [ ]:
for split in ("train", "dev", "test"):
    for name, docs in load_split(split).items():
        n = sum(len(d.gold_mentions) for d in docs)
        wide = sum(1 for d in docs for m in d.gold_mentions
                   if m["end"] - m["start"] + 1 > CONFIG["MAX_SPAN_WIDTH"])
        print(f"{name:9} {split:5} {len(docs):5} docs  {n:6} mentions  "
              f"({wide} wider than {CONFIG['MAX_SPAN_WIDTH']} tokens, cannot be predicted)")

## 3. Smoke test

Uses the example from the architecture diagram: *"Maya founded Northstar Labs. The
company makes biosensors."* The model should learn to find Maya (person),
Northstar Labs and The company (organization).

In [ ]:
def smoke_test(model_name, steps=60):
    torch.manual_seed(0)
    model = EntityIdentifier(model_name).to(DEVICE)
    d = model.encoder.hidden_size
    doc = document_from_text("Maya founded Northstar Labs. The company makes biosensors.",
                             ["person", "organization", "method"])
    doc.gold_mentions = [{"start": 0, "end": 0, "label": "person"},
                         {"start": 2, "end": 3, "label": "organization"},
                         {"start": 5, "end": 6, "label": "organization"}]

    out = model([doc])
    res = out["docs"][0]
    assert len(res["spans"]) == 52, len(res["spans"])
    assert res["word_vectors"].shape == (10, d)
    assert res["span_vectors"].shape == (52, 3 * d)
    assert res["logits"].shape == (52, 3)
    print(f"[ok] 52 candidate spans, span vectors {tuple(res['span_vectors'].shape)}, "
          f"initial loss {out['loss'].item():.4f}")

    long_words = ("Maya founded Northstar Labs . " * 250).split()
    assert model.encoder(long_words).shape == (len(long_words), d)
    print(f"[ok] long document of {len(long_words)} words encoded with sliding windows")

    assert model.represent_spans(res["word_vectors"], [(0, 9)]).shape == (1, 3 * d)
    print("[ok] represent_spans works for a 10-token span")

    opt = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": 2e-5},
        {"params": [p for n, p in model.named_parameters() if not n.startswith("encoder.")],
         "lr": 1e-3}])
    model.train()
    first = None
    for _ in range(steps):
        loss = model([doc])["loss"]
        opt.zero_grad(); loss.backward(); opt.step()
        first = loss.item() if first is None else first
    print(f"[ok] loss {first:.4f} -> {loss.item():.4f} after {steps} steps")

    pred = model.predict([doc])[0]
    for m in pred["mentions"]:
        print(f"     {m['text']!r:18} tokens {m['start']}-{m['end']}, {m['label']} ({m['score']:.2f})")
    got = {(m["start"], m["end"], m["label"]) for m in pred["mentions"]}
    want = {(m["start"], m["end"], m["label"]) for m in doc.gold_mentions}
    assert got == want, f"predicted {got}, expected {want}"
    assert pred["mention_vectors"].shape == (3, 3 * d)
    print("[ok] predictions match the gold mentions. All smoke tests passed.")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


smoke_test(CONFIG["MODEL_NAME"])